![image.png](https://i.imgur.com/a3uAqnb.png)

# **🌊 Flood Area Segmentation with Custom U-net & U-Net (Pretrained Encoder)**  
In this lab, we will:

✅ **Build a custom Dataset class** for flood segmentation  
✅ **Use `segmentation_models_pytorch (SMP)`** to load **U-Net** with a **pretrained encoder**  
✅ Train the model and evaluate its performance

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("faizalkarim/flood-area-segmentation")

print("Path to dataset files:", path)

100%|██████████| 107M/107M [00:01<00:00, 92.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/faizalkarim/flood-area-segmentation/versions/1


## **1️⃣ Dataset Class**

- The dataset consists of:
- **Images:** RGB flood images (`.jpg`)
- **Masks:** Corresponding segmentation masks (`.png`)
- **Metadata:** A CSV file (`metadata.csv`) mapping images to masks.
- The masks highlight the **water regions** in the images.
- We will create a **custom PyTorch Dataset class** to load images and masks.





In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0 (Use if Multiclass Segmentation)
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

def remap_mask_binary(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0 (Use if Binary Segmentation)
    mask_np = mask.numpy().squeeze()
    # Convert to binary: non-zero values become 1
    binary_mask = (mask_np != 0).astype(np.uint8)
    return torch.from_numpy(binary_mask).unsqueeze(0)

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
class FloodSegmentationDataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.metadata = pd.read_csv(csv_file)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, "Image", self.metadata.iloc[idx, 0])   # Use the csv to get paths
        mask_path = os.path.join(self.root_dir, "Mask", self.metadata.iloc[idx, 1])

        image = TODO..
        mask = TODO..   # Convert mask to grayscale (1 channel = binary segmentation)

        if self.transform:
            image = self.transform(TODO..)

        if self.target_transform:
            mask = self.target_transform(TODO..)

        # Replace mask values with remapped values
        mask = remap_mask_binary(mask)

        return TODO..      # In image classification datasets, we return image and label. Here, we return image and mask

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), TODO.. ),  # Keep segmentation masks intact
    transforms.PILToTensor(),

])

In [ ]:
## **🔹 Splitting the Dataset into Train & Test**

# Define dataset paths
csv_path = os.path.join(path, "metadata.csv")

# Read metadata CSV
metadata = pd.read_csv(csv_path)

# Split dataset into 80% train, 20% test
train_data, test_data = train_test_split(metadata, test_size=0.2, random_state=42, shuffle=True)

# Save split CSVs
train_data.to_csv(os.path.join(path, "train.csv"), index=False)
test_data.to_csv(os.path.join(path, "test.csv"), index=False)

In [ ]:
# Load training dataset
train_dataset = FloodSegmentationDataset(root_dir=path, csv_file=os.path.join(path, "train.csv"),
                                         transform=image_transforms, target_transform=mask_transforms)

# Load testing dataset
test_dataset = FloodSegmentationDataset(root_dir=path, csv_file=os.path.join(path, "test.csv"),
                                        transform=image_transforms, target_transform=mask_transforms)


# Create Train & Test DataLoaders
TODO..

# Check dataset sizes
TODO..

### Let's display some images

In [ ]:
import matplotlib.pyplot as plt

# Function to denormalize images (We cannot show normalized images. We have to reverse normalizaion first.)
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


## **2️⃣-1 Model Class**

We define a U-Net model with skip connections (residuals) implemented using concatenation.

![image.png](https://i.imgur.com/UVgm5kz.png)

# Custom UNet

## **3️⃣ Training and Validation Loops**

## **4️⃣ Running Training**
- We fine-tune the **pretrained U-Net model** on our flood segmentation dataset.
- We train the model using **Binary Cross Entropy (BCE) Loss**.

## **5️⃣ Visualizing Predictions**

- We compare **predicted flood masks** against **ground truth masks**.
- We visualize results for multiple test images.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

## **2️⃣-2 Model Class**

- We use **U-Net** from the **`segmentation_models_pytorch (SMP)`** library.
- The encoder (backbone) is **pretrained EfficientNet-B0** for better feature extraction.
- The decoder is **randomly initialized** and trained for segmentation.
- **Binary segmentation output (1 class for flood area)**  
- **Sigmoid activation** to output probability maps  
- **Binary Cross Entropy (BCE) Loss**  

# U-Net (Pretrained Encoder)

In [ ]:
#!pip install -q segmentation_models_pytorch

## **3️⃣ Training and Validation Loops**

## **4️⃣ Running Training**
- We fine-tune the **pretrained U-Net model** on our flood segmentation dataset.
- We train the model using **Binary Cross Entropy (BCE) Loss**.

### **🔹 Plot Training Loss Curve**


## **5️⃣ Visualizing Predictions**

- We compare **predicted flood masks** against **ground truth masks**.
- We visualize results for multiple test images.

### Contributed by: Mohamed Eltayeb